# Dependency Graph Analysis

Structural analysis of the build dependency graph: centrality metrics, layering, communities, and violations. Identifies architectural bottlenecks and coupling.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("../../data/processed")
PROJECT_ROOT = Path("../..")
INTERMEDIATE_DIR = Path("../../data/intermediate")

from buildanalysis.loading import BuildDataset
ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "axes.titlesize": 14})
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
from buildanalysis.types import BuildGraph
from buildanalysis.graph import (
    build_dependency_graph, compute_transitive_deps, compute_centrality_metrics,
    compute_layer_assignments, find_layer_violations, compute_graph_summary,
)
from buildanalysis.modularity import (
    detect_communities_louvain, hierarchical_clustering,
    compute_modularity_score, build_feature_configurations, compare_community_methods,
)
from buildanalysis.features import detect_thin_dependencies

## 1. Graph Summary

Overall structure of the build dependency graph.

In [ ]:
import networkx as nx

target_metrics = ds.target_metrics
edge_list = ds.edge_list

bg = build_dependency_graph(target_metrics, edge_list)

summary = compute_graph_summary(bg)

print("Build Dependency Graph Summary:")
print(f"  Targets (nodes): {bg.n_targets}")
print(f"  Dependencies (edges): {bg.n_edges}")
print(f"  Graph density: {nx.density(bg.graph):.4f}")
print(f"  Is DAG: {nx.is_directed_acyclic_graph(bg.graph)}")

components = list(nx.weakly_connected_components(bg.graph))
print(f"  Weakly connected components: {len(components)}")
print(f"  Largest component size: {len(max(components, key=len)) if components else 0}")

In [ ]:
print("\nDetailed Graph Metrics:")
print(f"  Average in-degree: {sum(dict(bg.graph.in_degree()).values()) / bg.n_targets:.2f}")
print(f"  Average out-degree: {sum(dict(bg.graph.out_degree()).values()) / bg.n_targets:.2f}")
print(f"  Max in-degree: {max(dict(bg.graph.in_degree()).values())}")
print(f"  Max out-degree: {max(dict(bg.graph.out_degree()).values())}")

if nx.is_directed_acyclic_graph(bg.graph):
    sorted_nodes = list(nx.topological_sort(bg.graph))
    if len(sorted_nodes) >= 2:
        try:
            dag_height = max((len(path) for path in nx.all_simple_paths(bg.graph, sorted_nodes[0], sorted_nodes[-1])), default=1)
            print(f"  DAG height (longest path): {dag_height}")
        except Exception:
            print(f"  DAG height: (disconnected or cycles detected)")

## 2. Centrality Analysis

Identify the most central targets in the dependency graph — architectural hubs.

In [ ]:
centrality = compute_centrality_metrics(bg)

print("Top 20 Targets by Betweenness Centrality:")
for target, score in centrality['betweenness'].sort_values(ascending=False).head(20).items():
    print(f"  {target:50s} {score:.4f}")

In [ ]:
print("\nTop 20 Targets by PageRank (importance in dependency graph):")
for target, score in centrality['pagerank'].sort_values(ascending=False).head(20).items():
    print(f"  {target:50s} {score:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

betweenness_values = centrality['betweenness'].values
in_degrees = centrality['in_degree'].values

ax.scatter(in_degrees, betweenness_values, alpha=0.6, s=50, color='steelblue')

top_betweenness = centrality['betweenness'].nlargest(10)
for target, score in top_betweenness.items():
    in_deg = centrality.loc[target, 'in_degree']
    ax.annotate(target.split('::')[-1], (in_deg, score), fontsize=8, alpha=0.7)

ax.set_xlabel('In-Degree (dependants)', fontsize=11)
ax.set_ylabel('Betweenness Centrality', fontsize=11)
ax.set_title('Centrality Analysis: In-Degree vs Betweenness', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Layer Analysis

Assign targets to dependency layers and analyse layering structure.

In [ ]:
layers = compute_layer_assignments(bg)
layer_map = layers.set_index("cmake_target")["layer"].to_dict()

print("Layer Distribution:")
layer_counts = layers["layer"].value_counts().sort_index()
for layer, count in layer_counts.items():
    print(f"  Layer {layer:2d}: {count:4d} targets")

print(f"\nTotal layers (depth): {layer_counts.index.max() + 1}")
print(f"Average targets per layer: {layer_counts.mean():.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

layer_counts.plot(kind='bar', ax=ax, color='steelblue', alpha=0.7)
ax.set_xlabel('Layer', fontsize=11)
ax.set_ylabel('Number of Targets', fontsize=11)
ax.set_title('Distribution of Targets Across Dependency Layers', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
max_layer = layers["layer"].max()
max_depth_targets = layers[layers["layer"] == max_layer]["cmake_target"].tolist()
print(f"Leaf Targets (maximum depth layer):")
print(f"  Count: {len(max_depth_targets)}")
print(f"  Examples:")
for target in max_depth_targets[:10]:
    print(f"    {target}")

## 4. Layer Violations

Detect edges that skip layers (violation of clean layering).

In [ ]:
violations = find_layer_violations(bg, layers)

print(f"Layer Violations (skip-layer edges):")
print(f"  Total violations: {len(violations)}")

if len(violations) > 0:
    print(f"\nViolation Examples (first 20):")
    for _, row in violations.head(20).iterrows():
        layers_skipped = abs(row['dep_layer'] - row['source_layer'])
        print(f"  {row['source']:40s} → {row['dependency']:40s} ({row['violation_type']}, skip {layers_skipped} layer(s))")
else:
    print(f"\nNo layer violations found! Graph has clean layering.")

## 5. Community Detection

Detect natural clusters (communities) in the dependency graph using Louvain algorithm.

In [ ]:
communities_df = detect_communities_louvain(bg)
community_map = communities_df.set_index("cmake_target")["community"].to_dict()
n_communities = communities_df["community"].nunique()

# Use compute_modularity_score for richer metrics
mod_score = compute_modularity_score(bg, communities_df)

print(f"Community Detection (Louvain):")
print(f"  Number of communities: {mod_score['n_communities']}")
print(f"  Modularity score: {mod_score['graph_modularity']:.4f}")
print(f"  Inter-community edge fraction: {mod_score['inter_community_edge_fraction']:.4f}")
print(f"  Average self-containment: {mod_score['avg_self_containment']:.4f}")
print(f"  Community size range: {mod_score['min_community_size']} - {mod_score['max_community_size']}")

community_sizes = communities_df["community"].value_counts().sort_values(ascending=False)
print(f"\nCommunity Sizes:")
for community_id, size in community_sizes.head(20).items():
    print(f"  Community {community_id:3d}: {size:4d} targets")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

community_sizes.head(20).plot(kind='bar', ax=ax, color='seagreen', alpha=0.7)
ax.set_xlabel('Community ID', fontsize=11)
ax.set_ylabel('Number of Targets', fontsize=11)
ax.set_title('Top 20 Communities by Size', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
print(f"\nTop 5 Largest Communities (members):")
for community_id in community_sizes.head(5).index:
    members = communities_df[communities_df["community"] == community_id]["cmake_target"].tolist()
    print(f"\nCommunity {community_id} ({len(members)} targets):")
    for member in members[:15]:
        print(f"  {member}")
    if len(members) > 15:
        print(f"  ... and {len(members) - 15} more")

## 6. Transitive Dependencies

Analyse the scope of transitive dependencies (all indirect dependencies).

In [ ]:
transitive_deps = compute_transitive_deps(bg)

transitive_counts = transitive_deps["n_transitive_deps"].values

print(f"Transitive Dependency Analysis:")
print(f"  Targets with transitive deps: {(transitive_deps['n_transitive_deps'] > 0).sum()}")
print(f"  Mean transitive deps per target: {np.mean(transitive_counts):.1f}")
print(f"  Max transitive deps (single target): {max(transitive_counts)}")
print(f"  Median transitive deps: {np.median(transitive_counts):.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(transitive_counts, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Number of Transitive Dependencies', fontsize=11)
ax.set_ylabel('Number of Targets', fontsize=11)
ax.set_title('Distribution of Transitive Dependency Counts', fontsize=12, fontweight='bold')
ax.axvline(np.mean(transitive_counts), color='red', linestyle='--', label=f'Mean: {np.mean(transitive_counts):.1f}')
ax.axvline(np.median(transitive_counts), color='orange', linestyle='--', label=f'Median: {np.median(transitive_counts):.1f}')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
direct_dep_counts = {n: bg.graph.out_degree(n) for n in bg.graph.nodes()}
direct_counts = list(direct_dep_counts.values())

top_transitive = transitive_deps.nlargest(20, "n_transitive_deps")

print(f"\nTop 20 Targets by Transitive Dependency Count:")
for _, row in top_transitive.iterrows():
    target = row["cmake_target"]
    n_trans = int(row["n_transitive_deps"])
    n_direct = int(row["n_direct_deps"])
    print(f"  {target:50s} {n_trans:4d} transitive (+ {n_direct:2d} direct)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

td_direct = transitive_deps["n_direct_deps"].values
td_trans = transitive_deps["n_transitive_deps"].values

ax.scatter(td_direct, td_trans, alpha=0.6, s=50, color='steelblue')

for _, row in top_transitive.head(10).iterrows():
    ax.annotate(row["cmake_target"].split('::')[-1], (row["n_direct_deps"], row["n_transitive_deps"]), fontsize=8, alpha=0.7)

ax.set_xlabel('Direct Dependencies (out-degree)', fontsize=11)
ax.set_ylabel('Transitive Dependencies', fontsize=11)
ax.set_title('Direct vs Transitive Dependency Scope', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Save Intermediates

Persist graph analysis results for downstream use.

In [ ]:
centrality_out = centrality[['betweenness', 'pagerank', 'in_degree', 'out_degree']].reset_index()
ds.save_intermediate("centrality_metrics", centrality_out)
print(f"Saved centrality metrics ({len(centrality_out)} rows, columns: {list(centrality_out.columns)})")

ds.save_intermediate("transitive_deps", transitive_deps)
print(f"Saved transitive deps ({len(transitive_deps)} rows)")

ds.save_intermediate("layer_assignments", layers)
print(f"Saved layer assignments")

ds.save_intermediate("community_assignments", communities_df)
print(f"Saved community assignments")

## 8. Community Method Comparison

Compare Louvain and hierarchical clustering results side by side.

In [ ]:
# Hierarchical clustering for comparison
from scipy.cluster.hierarchy import fcluster
Z, nodes = hierarchical_clustering(bg)
hier_labels = fcluster(Z, t=n_communities, criterion="maxclust") - 1
hier_communities = pd.DataFrame({"cmake_target": nodes, "community": hier_labels})

comparison = compare_community_methods(bg, {"louvain": communities_df, "hierarchical": hier_communities})
print("Community Method Comparison:")
print(comparison.to_string(index=False))

# Feature configurations — build scope per community
timing_for_features = target_metrics[["cmake_target", "total_build_time_ms"]].copy()
feature_configs = build_feature_configurations(bg, communities_df, timing=timing_for_features)

print(f"\nFeature Configurations (build scope per community):")
display_cols = [c for c in ["feature_id", "own_targets", "transitive_deps", "total_build_set",
                             "build_fraction", "estimated_build_time_ms"] if c in feature_configs.columns]
print(feature_configs[display_cols].to_string(index=False))

ds.save_intermediate("feature_configurations_louvain", feature_configs)
print(f"\nSaved feature_configurations_louvain.parquet")

## 9. Thin Dependencies

Identify dependency edges where the depending target uses only a small fraction of the depended-on target's headers — candidates for narrowing to interface libraries.

In [ ]:
file_metrics = ds.file_metrics
if "header_tree" in file_metrics.columns:
    header_data = file_metrics[["source_file", "cmake_target", "header_tree"]].dropna(subset=["header_tree"])
    thin_deps = detect_thin_dependencies(bg.graph, header_data, thinness_threshold=0.1)

    print(f"Thin Dependencies (thinness ratio <= 10%):")
    print(f"  Found: {len(thin_deps)} thin edges out of {bg.n_edges} total")
    if len(thin_deps) > 0:
        print(f"\nTop 20 thinnest dependencies:")
        for _, row in thin_deps.nsmallest(20, "thinness_ratio").iterrows():
            print(f"  {row['depending_target']:40s} -> {row['depended_target']:40s} "
                  f"uses {row['used_headers']}/{row['total_headers']} headers ({row['thinness_ratio']:.1%})")

        ds.save_intermediate("thin_dependencies", thin_deps)
        print(f"\nSaved thin_dependencies.parquet ({len(thin_deps)} rows)")
else:
    print("header_tree column not available in file_metrics — skipping thin dependency detection.")